# Time Series Decomposition

A simple time series decomposition. 

The data uses is the classic Box & Jenkins airline data, comprising monthly totals (in thousands) of international airline passengers, 1949 to 1960. 

References:
- https://pkg.robjhyndman.com/fma/reference/airpass.html
- https://machinelearningmastery.com/decompose-time-series-data-trend-seasonality/

### Step 1: Imports

In [1]:
# Packages

# Data manipulation
import pandas as pd
import copy

# Modelling
from statsmodels.tsa.seasonal import seasonal_decompose

# Plotting
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Step 2: Read and Display Data

In [2]:
# Read
df_air=pd.read_csv('air.txt')

# Information
print(df_air.info())

# Head and Tail
display(df_air)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Month       144 non-null    object
 1   Passengers  144 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 2.4+ KB
None


,Month,Passengers
0,1949-01,112
1,1949-02,118
2,1949-03,132
3,1949-04,129
4,1949-05,121
...,...,...
139,1960-08,606
140,1960-09,508
141,1960-10,461
142,1960-11,390


### Step 3: Plot Original Time Series

In [3]:
# Plot
fig = px.line(data_frame=df_air, 
              x='Month', 
              y='Passengers',
              width=800,
              height=500,
              range_y =[0,700],
              title='International Airline Passenger Numbers (thousands)')
fig.update_layout(title_x=0.5, title_y=0.85)
fig.show()

C:\Users\markt\AppData\Roaming\Python\Python312\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




### Step 4: Multiplicative 12-Month Seasonal Decomposition

In [4]:
# Seasonally adust
result = seasonal_decompose(df_air.Passengers, period=12, model='multiplicative')

# Compile results
df_result=pd.DataFrame({'Month':df_air.Month,
                       'observed':result.observed,
                       'seasonal':result.seasonal,
                       'trend':result.trend,
                       'resid':result.resid})

# Fitted values as Trend*Seasonal
df_result['fitted']=df_result.trend * df_result.seasonal

### Step 5: Print the Decomposition

In [5]:
# Print results
fig = make_subplots(rows=2, cols=2)

fig.add_trace(go.Scatter(x=df_result.Month, y=df_result.observed, mode='lines', name='Observed'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_result.Month, y=df_result.seasonal, mode='lines', name='Seasonal'), row=1, col=2)
fig.add_trace(go.Scatter(x=df_result.Month, y=df_result.trend,    mode='lines', name='Trend'), row=2, col=1)
fig.add_trace(go.Scatter(x=df_result.Month, y=df_result.resid,    mode='lines', name='Residual'), row=2, col=2)

fig.update_layout(height=720, width=960, title_text='Airline Passenger Numbers (Thousands)')
fig.update_layout(title_x=0.5, title_y=0.90)
fig.show()

### Step 6: Compare Original and Fitted Values (Trend*Seasonal)

In [6]:
# Plot - Fitted and Original
df_result2=pd.melt(df_result, id_vars='Month', value_vars=['observed','fitted'], var_name='Legend', value_name='Passengers')
fig = px.line(data_frame=df_result2, 
              x='Month', 
              y='Passengers',
              color='Legend',
              width=800,
              height=500,
              #range_y =[0,700],
              title='Airline Passenger Numbers (thousands)')
fig.update_layout(title_x=0.5, title_y=0.85)
fig.show()